In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("examsgovt/sme-financial-decision-risk-prediction-dataset")

print("Path to dataset files:", path)

c:\Users\marks\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 733k/733k [00:00<00:00, 897kB/s]

Extracting files...
Path to dataset files: C:\Users\marks\.cache\kagglehub\datasets\examsgovt\sme-financial-decision-risk-prediction-dataset\versions\2


# SME Financial Distress NL2SQL with AlloyDB AI

**Use case:** Querying SME financial distress drivers across sectors and revenue bands using natural language.

This notebook implements the assignment end-to-end:
1. Load a custom dataset (SME Financial Decision Risk Prediction).
2. Create/modify schema objects in AlloyDB for PostgreSQL.
3. Configure AlloyDB AI natural language.
4. Convert NL question -> SQL -> execute SQL -> return results.

In [ ]:
# If needed, install dependencies for PostgreSQL connection and data loading.
# You can skip this cell if packages are already available.
%pip -q install pandas psycopg[binary] python-dotenv

In [2]:
from pathlib import Path
import os
import re
import pandas as pd

# The first notebook cell stores Kaggle dataset folder in `path`.
base_path = Path(path)
csv_candidates = sorted(base_path.glob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"No CSV files found in {base_path}")

csv_path = csv_candidates[0]
print(f"Using CSV file: {csv_path}")

df_raw = pd.read_csv(csv_path)
print(f"Rows: {len(df_raw):,} | Columns: {len(df_raw.columns)}")
df_raw.head()

Using CSV file: C:\Users\marks\.cache\kagglehub\datasets\examsgovt\sme-financial-decision-risk-prediction-dataset\versions\2\financial_dataset_SME.csv
Rows: 15,106 | Columns: 33


,Has_Financial_Questions,SME_Age,SME_Type,Industry_Sector,SME_Size_Category,Literacy_Accounting,Literacy_Budgeting,Literacy_Investment_Evaluation,Literacy_Credit_Knowledge,Risk_Awareness,...,Decision_Capital_Allocation,Decision_CashFlow_Management,Analysis_Accounting_Tools,Analysis_Financial_Ratios,Analysis_Forecasting,Analysis_Benchmarking,Liquidity_Stability,Uses_Digital_Finance,Annual_Revenue_Category,Financial_Distress
0,YES,21.0,3.0,1.0,1.0,3.0,3.0,3.0,6.0,0.0,...,6.0,5.0,3.0,5.0,3.0,5.0,6.0,0.0,2.0,0
1,YES,29.0,2.0,2.0,3.0,2.0,2.0,3.0,2.0,0.0,...,0.0,3.0,2.0,8.0,3.0,5.0,3.0,0.0,2.0,0
2,YES,29.0,2.0,1.0,3.0,3.0,3.0,5.0,5.0,7.0,...,1.0,3.0,3.0,2.0,5.0,5.0,7.0,0.0,4.0,1
3,YES,6.0,2.0,2.0,2.0,4.0,4.0,5.0,6.0,9.0,...,3.0,5.0,4.0,7.0,5.0,5.0,4.0,0.0,2.0,1
4,YES,12.0,2.0,3.0,4.0,3.0,3.0,3.0,6.0,9.0,...,4.0,3.0,3.0,5.0,3.0,5.0,6.0,0.0,2.0,1


In [5]:
# Normalize column names so PostgreSQL identifiers are consistent.
def normalize_col(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

column_map = {c: normalize_col(c) for c in df_raw.columns}
df = df_raw.rename(columns=column_map).copy()

# Enforce stable dtypes before loading into PostgreSQL.
int_like_cols = [
    "sme_age",
    "sme_type",
    "industry_sector",
    "sme_size_category",
    "uses_digital_finance",
    "financial_distress",
]
for col in int_like_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

print("Column mapping (first 12):")
for old, new in list(column_map.items())[:12]:
    print(f"{old} -> {new}")

df.head()

Column mapping (first 12):
Has_Financial_Questions -> has_financial_questions
SME_Age -> sme_age
SME_Type -> sme_type
Industry_Sector -> industry_sector
SME_Size_Category -> sme_size_category
Literacy_Accounting -> literacy_accounting
Literacy_Budgeting -> literacy_budgeting
Literacy_Investment_Evaluation -> literacy_investment_evaluation
Literacy_Credit_Knowledge -> literacy_credit_knowledge
Risk_Awareness -> risk_awareness
Risk_Evaluation -> risk_evaluation
Risk_Mitigation_Strategies -> risk_mitigation_strategies


,has_financial_questions,sme_age,sme_type,industry_sector,sme_size_category,literacy_accounting,literacy_budgeting,literacy_investment_evaluation,literacy_credit_knowledge,risk_awareness,...,decision_capital_allocation,decision_cashflow_management,analysis_accounting_tools,analysis_financial_ratios,analysis_forecasting,analysis_benchmarking,liquidity_stability,uses_digital_finance,annual_revenue_category,financial_distress
0,YES,21,3,1,1,3.0,3.0,3.0,6.0,0.0,...,6.0,5.0,3.0,5.0,3.0,5.0,6.0,0,2.0,0
1,YES,29,2,2,3,2.0,2.0,3.0,2.0,0.0,...,0.0,3.0,2.0,8.0,3.0,5.0,3.0,0,2.0,0
2,YES,29,2,1,3,3.0,3.0,5.0,5.0,7.0,...,1.0,3.0,3.0,2.0,5.0,5.0,7.0,0,4.0,1
3,YES,6,2,2,2,4.0,4.0,5.0,6.0,9.0,...,3.0,5.0,4.0,7.0,5.0,5.0,4.0,0,2.0,1
4,YES,12,2,3,4,3.0,3.0,3.0,6.0,9.0,...,4.0,3.0,3.0,5.0,3.0,5.0,6.0,0,2.0,1


In [6]:
# Quick profiling for presentation evidence.
profile = {
    "row_count": len(df),
    "col_count": len(df.columns),
    "null_cells": int(df.isna().sum().sum()),
    "target_dist": df["financial_distress"].value_counts(dropna=False).to_dict(),
}
profile

{'row_count': 15106,
 'col_count': 33,
 'null_cells': 0,
 'target_dist': {0: 10000, 1: 5106}}

## AlloyDB Connection Setup

Set these environment variables before running the next cells:
- `ALLOYDB_HOST`
- `ALLOYDB_PORT` (default `5432`)
- `ALLOYDB_DB` (for example `sme_demo`)
- `ALLOYDB_USER`
- `ALLOYDB_PASSWORD`

If you are using Cloud Shell or local machine, use the AlloyDB Auth Proxy / private connectivity as needed.

In [ ]:
import io
import psycopg
from psycopg.rows import dict_row

DB_CFG = {
    "host": os.getenv("ALLOYDB_HOST", ""),
    "port": int(os.getenv("ALLOYDB_PORT", "5432")),
    "dbname": os.getenv("ALLOYDB_DB", ""),
    "user": os.getenv("ALLOYDB_USER", ""),
    "password": os.getenv("ALLOYDB_PASSWORD", ""),
    "sslmode": os.getenv("ALLOYDB_SSLMODE", "require"),
}

missing = [k for k, v in DB_CFG.items() if k in {"host", "dbname", "user", "password"} and not v]
if missing:
    print("Missing env vars for DB connection:", missing)
else:
    print("All required DB connection env vars are set.")


def get_conn():
    return psycopg.connect(**DB_CFG, row_factory=dict_row)


def run_sql(sql: str, params=None, fetch=True):
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            if fetch and cur.description:
                return cur.fetchall()
            return None

In [ ]:
# Create schema and base table in AlloyDB.
create_schema_sql = """
CREATE SCHEMA IF NOT EXISTS sme_risk;
"""

# Keep all fields as numeric/text aligned with source, then add custom derived column.
create_table_sql = """
CREATE TABLE IF NOT EXISTS sme_risk.sme_financial (
    has_financial_questions text,
    sme_age int,
    sme_type int,
    industry_sector int,
    sme_size_category int,
    literacy_accounting numeric,
    literacy_budgeting numeric,
    literacy_investment_evaluation numeric,
    literacy_credit_knowledge numeric,
    risk_awareness numeric,
    risk_evaluation numeric,
    risk_mitigation_strategies numeric,
    risk_taking_willingness numeric,
    assessment_data_driven numeric,
    assessment_expert_consultation numeric,
    assessment_scenario_analysis numeric,
    assessment_internal_evaluation numeric,
    decision_autonomy numeric,
    decision_consultation numeric,
    decision_financial_advisor numeric,
    decision_strategic_alignment numeric,
    decision_investment_choices numeric,
    decision_loan_approval numeric,
    decision_capital_allocation numeric,
    decision_cashflow_management numeric,
    analysis_accounting_tools numeric,
    analysis_financial_ratios numeric,
    analysis_forecasting numeric,
    analysis_benchmarking numeric,
    liquidity_stability numeric,
    uses_digital_finance int,
    annual_revenue_category text,
    financial_distress int,
    distress_label text GENERATED ALWAYS AS (
        CASE WHEN financial_distress = 1 THEN 'Distressed' ELSE 'Stable' END
    ) STORED
);
"""

index_sql = """
CREATE INDEX IF NOT EXISTS idx_sme_sector ON sme_risk.sme_financial(industry_sector);
CREATE INDEX IF NOT EXISTS idx_sme_revenue ON sme_risk.sme_financial(annual_revenue_category);
CREATE INDEX IF NOT EXISTS idx_sme_digital_finance ON sme_risk.sme_financial(uses_digital_finance);
CREATE INDEX IF NOT EXISTS idx_sme_distress ON sme_risk.sme_financial(financial_distress);
"""

view_sql = """
CREATE OR REPLACE VIEW sme_risk.vw_distress_rate_by_segment AS
SELECT
    industry_sector,
    sme_size_category,
    annual_revenue_category,
    COUNT(*) AS total_smes,
    AVG(financial_distress::numeric) AS distress_rate
FROM sme_risk.sme_financial
GROUP BY industry_sector, sme_size_category, annual_revenue_category;
"""

for sql_block in [create_schema_sql, create_table_sql, index_sql, view_sql]:
    run_sql(sql_block, fetch=False)

print("Schema, table, indexes, and view are ready.")

In [ ]:
# Load DataFrame into AlloyDB table using COPY.
columns = list(df.columns)
copy_cols = ",".join(columns)

csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False, header=False)
csv_buffer.seek(0)

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE sme_risk.sme_financial;")
        with cur.copy(f"COPY sme_risk.sme_financial ({copy_cols}) FROM STDIN WITH CSV") as copy:
            copy.write(csv_buffer.read())
    conn.commit()

row_count = run_sql("SELECT COUNT(*) AS n FROM sme_risk.sme_financial;")
row_count

In [ ]:
# Basic validation query for presentation screenshot.
run_sql("""
SELECT distress_label, COUNT(*) AS n
FROM sme_risk.sme_financial
GROUP BY distress_label
ORDER BY distress_label;
""")

## Configure AlloyDB AI Natural Language

This section enables and tunes `alloydb_ai_nl` for the SME dataset.

In [ ]:
# 1) Enable extension and create NL config.
run_sql("CREATE EXTENSION IF NOT EXISTS alloydb_ai_nl CASCADE;", fetch=False)

run_sql("SELECT alloydb_ai_nl.g_create_configuration('sme_finance_cfg');", fetch=False)
run_sql("""
SELECT alloydb_ai_nl.g_manage_configuration(
    operation => 'register_schema',
    configuration_id_in => 'sme_finance_cfg',
    schema_names_in => '{sme_risk}'
);
""", fetch=False)

print("alloydb_ai_nl extension enabled and configuration created.")

In [ ]:
# 2) Add domain context and apply generated schema context.
run_sql("""
SELECT alloydb_ai_nl.g_manage_configuration(
    'add_general_context',
    'sme_finance_cfg',
    general_context_in => '{"In this dataset, financial_distress is binary: 0 means Stable, 1 means Distressed."}'
);
""", fetch=False)

run_sql("""
SELECT alloydb_ai_nl.g_manage_configuration(
    'add_general_context',
    'sme_finance_cfg',
    general_context_in => '{"uses_digital_finance uses values 0 and 1 where 1 means digital finance is used."}'
);
""", fetch=False)

run_sql("SELECT alloydb_ai_nl.generate_schema_context('sme_finance_cfg');", fetch=False)
run_sql("SELECT alloydb_ai_nl.apply_generated_schema_context('sme_finance_cfg', TRUE);", fetch=False)

print("General context and schema context applied.")

In [ ]:
# 3) Add curated templates aligned to SME risk analysis.
run_sql("""
SELECT alloydb_ai_nl.add_template(
  nl_config_id => 'sme_finance_cfg',
  intent => 'What is the distress rate by industry sector?',
  sql => $$
    SELECT industry_sector,
           AVG(financial_distress::numeric) AS distress_rate,
           COUNT(*) AS total_smes
    FROM sme_risk.sme_financial
    GROUP BY industry_sector
    ORDER BY distress_rate DESC
  $$,
  check_intent => TRUE
);
""", fetch=False)

run_sql("""
SELECT alloydb_ai_nl.add_template(
  nl_config_id => 'sme_finance_cfg',
  intent => 'Compare distress rate for SMEs that use digital finance versus those that do not.',
  sql => $$
    SELECT uses_digital_finance,
           AVG(financial_distress::numeric) AS distress_rate,
           COUNT(*) AS total_smes
    FROM sme_risk.sme_financial
    GROUP BY uses_digital_finance
    ORDER BY uses_digital_finance
  $$,
  check_intent => TRUE
);
""", fetch=False)

run_sql("""
SELECT alloydb_ai_nl.add_template(
  nl_config_id => 'sme_finance_cfg',
  intent => 'Which annual revenue category has the highest number of distressed SMEs?',
  sql => $$
    SELECT annual_revenue_category,
           COUNT(*) FILTER (WHERE financial_distress = 1) AS distressed_count,
           COUNT(*) AS total_smes
    FROM sme_risk.sme_financial
    GROUP BY annual_revenue_category
    ORDER BY distressed_count DESC
  $$,
  check_intent => TRUE
);
""", fetch=False)

print("Templates added.")

In [ ]:
# 4) Associate concept types and build value index.
run_sql("""
SELECT alloydb_ai_nl.associate_concept_type(
    column_names_in => 'sme_risk.sme_financial.annual_revenue_category',
    concept_type_in => 'generic_entity_name',
    nl_config_id_in => 'sme_finance_cfg'
);
""", fetch=False)

run_sql("""
SELECT alloydb_ai_nl.create_value_index(
    nl_config_id_in => 'sme_finance_cfg'
);
""", fetch=False)

print("Concept association and value index created.")

## NL Question -> Generated SQL -> Results

Use your own natural-language questions here to satisfy originality criteria.

In [ ]:
def generate_sql_from_nl(question: str):
    sql_row = run_sql(
        """
        SELECT alloydb_ai_nl.get_sql(
            nl_config_id => 'sme_finance_cfg',
            nl_question => %(q)s
        ) ->> 'sql' AS generated_sql;
        """,
        params={"q": question},
    )
    return sql_row[0]["generated_sql"]


def ask(question: str):
    generated_sql = generate_sql_from_nl(question)
    print("NL Question:", question)
    print("\nGenerated SQL:\n", generated_sql)
    result = run_sql(generated_sql)
    return pd.DataFrame(result)

In [ ]:
# Original question example (required by assignment).
q1 = "Which annual revenue category has the highest financial distress rate among SMEs?"
ask(q1).head(20)

In [ ]:
# Additional example questions for robustness and presentation screenshots.
questions = [
    "Compare financial distress rate between SMEs that use digital finance and those that do not.",
    "Show the top 5 industry sectors with the highest distress rate.",
    "For low annual revenue SMEs, what is the distress rate by liquidity stability score?",
]

for q in questions:
    print("=" * 100)
    display(ask(q).head(10))

## Presentation Evidence Checklist

Capture screenshots for these outputs:
1. Dataset load summary (`Rows`, `Columns`, target distribution).
2. Custom schema objects created (`sme_risk.sme_financial`, generated column, and view).
3. AlloyDB AI natural language configuration steps completed.
4. Original NL question, generated SQL text, and resulting output table.
5. At least one additional NL question result for robustness.

Originality statement for slide:
- Dataset is SME Financial Decision Risk Prediction (not lab default).
- Queries were authored for SME risk analysis use cases.

In [ ]:
# Optional: quick evidence table for slide content.
evidence_rows = [
    {"criterion": "Custom dataset", "evidence": "Loaded SME CSV with 15,106 rows"},
    {"criterion": "Schema created/modified", "evidence": "Created sme_risk.sme_financial + generated distress_label + view"},
    {"criterion": "AlloyDB AI NL enabled", "evidence": "Enabled alloydb_ai_nl and configured sme_finance_cfg"},
    {"criterion": "Original NL query", "evidence": "Which annual revenue category has the highest financial distress rate among SMEs?"},
    {"criterion": "NL -> SQL -> Results", "evidence": "ask(question) prints generated SQL and returns executed result DataFrame"},
]

pd.DataFrame(evidence_rows)